# 대구광역시 응급의료 취약성 포트폴리오 EDA
본 노트북은 단일 `policy_release.json`에 묶인 검증 분석본의 분포와 변수 관계를 점검합니다.
실행을 위해서는 `pandas`, `matplotlib`, `seaborn` 라이브러리가 필요합니다.

In [ ]:
import hashlib
import json
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
import seaborn as sns

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
font_candidates = ('Malgun Gothic', 'AppleGothic', 'Noto Sans CJK KR', 'Noto Sans KR', 'NanumGothic')
selected_font = next((font for font in font_candidates if font in available_fonts), 'DejaVu Sans')
plt.rc('font', family=selected_font)
plt.rcParams['axes.unicode_minus'] = False

def payload_sha256(value):
    encoded = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()

release_path = Path('../frontend/public/data/policy_release.json')
with open(release_path, encoding='utf-8') as f:
    release = json.load(f)
metadata = release['metadata']
vul_data = release['vulnerability']
expected_routes = metadata['district_count'] * (metadata['resource_count'] + metadata['candidate_count'])
assert metadata['district_count'] == len(vul_data['features']) == 150
assert metadata['resource_count'] == len(release['hospitals']) == 25
assert metadata['candidate_count'] == len(release['candidates']) == 9
assert metadata['route_count'] == metadata['successful_route_count'] == expected_routes == 5100
assert metadata['missing_route_count'] == 0
for name in ('hospitals', 'vulnerability', 'candidates', 'candidate_trace', 'optimization'):
    assert payload_sha256(release[name]) == metadata['content_sha256'][name], f'{name} SHA-256 불일치'

features = []
for feat in vul_data['features']:
    props = feat['properties']
    features.append({
        'adm_nm': props.get('adm_nm', ''),
        'pop_vul': props.get('취약인구', 0),
        'road_eta': props.get('travel_time_minutes', 0),
        'vdi': props.get('vulnerability_index', 0),
        'nearest_tier': props.get('nearest_hospital_tier', 3)
    })
df = pd.DataFrame(features)
df.head()

### 기초 통계 파악

In [ ]:
df.describe()

### VDI 산식 구성요소와 민감도 점검
VDI는 `ln(1 + 일반 차량 ETA) × 취약인구`이므로 아래 상관은 독립적인 인과 발견이 아니라 산식 구성요소 민감도 점검입니다.

In [ ]:
corr_cols = ['pop_vul', 'road_eta', 'vdi']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm')
plt.title('VDI formula component sensitivity')
plt.show()